# E1 only: conditional hidden-state comparison

This notebook runs only the nested E1 analysis: N, N+H, O, O+H, and H. It reuses frozen Experiment 1 activation shards and the selected-layer artifact already stored in Google Drive. It does not load the language model, extract activations, refit the original layer sweep, run interventions, or build the paper.

A GPU is not required for this analysis. A Colab high-RAM runtime is recommended because the frozen activation shards occupy roughly 2 GiB before analysis workspaces are allocated.

## 1. Load the repository

Set `REPOSITORY_REF` to the branch or tag containing the E1 implementation before running this cell. If the notebook is already running from a repository checkout, that checkout is used directly.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/sagnikc395/tracing-mathematical-error-detection-in-language-models.git"
REPOSITORY_REF = "main"  # Change if E1 is on another branch.
REPOSITORY_NAME = "tracing-mathematical-error-detection-in-language-models"

working_directory = Path.cwd()
if (working_directory / "pyproject.toml").exists():
    repository_directory = working_directory
elif (working_directory.parent / "pyproject.toml").exists():
    repository_directory = working_directory.parent
else:
    repository_directory = Path("/content") / REPOSITORY_NAME
    if not (repository_directory / "pyproject.toml").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                REPOSITORY_REF,
                REPOSITORY_URL,
                str(repository_directory),
            ],
            check=True,
        )

os.chdir(repository_directory)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
    check=True,
)
print(f"Repository: {repository_directory}")
print(f"Python: {sys.version.split()[0]}")

## 2. Mount Drive and identify the frozen inputs

Edit `EXPERIMENT1_DIR` if your previous notebook used a different Drive folder. The directory must directly contain both `activation_shards/` and `probes/directions.npz`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/math-error-tracing")
EXPERIMENT1_DIR = (
    DRIVE_ROOT / "artifacts/qwen2.5-math-1.5b-a100-bf16"
)
DATA_PATH = DRIVE_ROOT / "data/processbench.jsonl"
OUTPUT_DIR = DRIVE_ROOT / "artifacts/experiment3-extended"

print(f"Frozen Experiment 1 directory: {EXPERIMENT1_DIR}")
print(f"ProcessBench data: {DATA_PATH}")
print(f"E1 output directory: {OUTPUT_DIR / 'conditional_hidden_state'}")

## 3. Validate the shard set

Every shard must have an activation array (`.npy`), aligned metadata (`.csv`), and manifest (`.json`). The selected layer is read from the frozen probe artifact. The dataset hash is checked against the extraction identity before any model is fit.

In [ ]:
import hashlib
import json

import numpy as np

SHARD_DIR = EXPERIMENT1_DIR / "activation_shards"
DIRECTION_PATH = EXPERIMENT1_DIR / "probes/directions.npz"
IDENTITY_PATH = EXPERIMENT1_DIR / "extraction_identity.json"

for required_path in (SHARD_DIR, DIRECTION_PATH, IDENTITY_PATH, DATA_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Required frozen input is missing: {required_path}")

array_paths = sorted(SHARD_DIR.glob("shard_*.npy"))
if not array_paths:
    raise FileNotFoundError(f"No activation shards found under {SHARD_DIR}")
incomplete = [
    path.stem
    for path in array_paths
    if not path.with_suffix(".csv").exists()
    or not path.with_suffix(".json").exists()
]
if incomplete:
    raise RuntimeError(f"Incomplete shard triplets: {incomplete[:5]}")

identity = json.loads(IDENTITY_PATH.read_text())
dataset_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
if identity.get("dataset_sha256") != dataset_sha256:
    raise RuntimeError(
        "The ProcessBench file does not match the dataset used for shard extraction."
    )
if identity.get("dtype") != "bfloat16":
    raise RuntimeError(f"Expected the frozen BF16 run, found {identity.get('dtype')!r}")

directions = np.load(DIRECTION_PATH)
selected_layer = int(directions["selected_layer"])
shard_bytes = sum(path.stat().st_size for path in array_paths)
print(f"Complete shard triplets: {len(array_paths)}")
print(f"Activation-array size: {shard_bytes / 2**30:.2f} GiB")
print(f"Frozen selected layer: {selected_layer}")
print(json.dumps(identity, indent=2))

## 4. Resolve the E1 configuration

Only filesystem paths are changed from `configs/experiment3.yaml`. The C grid, bootstrap count, seed, confidence level, feature definitions, and practical AUROC margin remain frozen in the repository configuration.

In [ ]:
import yaml

with Path("configs/experiment3.yaml").open(encoding="utf-8") as handle:
    e1_config = yaml.safe_load(handle)
e1_config["experiment1_dir"] = str(EXPERIMENT1_DIR)
e1_config["data_path"] = str(DATA_PATH)
e1_config["output_dir"] = str(OUTPUT_DIR)
e1_config["counterfactual_patching"]["pairs_path"] = str(
    DRIVE_ROOT / "data/counterfactual_pairs.jsonl"
)

CONFIG_PATH = Path("/content/conditional_hidden_state.yaml")
CONFIG_PATH.write_text(
    yaml.safe_dump(e1_config, sort_keys=False), encoding="utf-8"
)
print(CONFIG_PATH.read_text())

## 5. Run E1

This is the only experiment command in the notebook. It loads the frozen shards, fits all five conditions, evaluates the untouched test partition, and writes its outputs directly to Drive. No existing Experiment 1 artifact is modified.

In [ ]:
from datetime import datetime, timezone
from time import monotonic

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUTPUT_DIR / "conditional_hidden_state.log"
command = [
    sys.executable,
    "-m",
    "tracing_math.experiment3.cli",
    "--config",
    str(CONFIG_PATH),
    "fit-conditional-hidden-state",
]
print("$", " ".join(command), flush=True)
started = monotonic()
started_at = datetime.now(timezone.utc).isoformat()
with LOG_PATH.open("a", encoding="utf-8", buffering=1) as log_file:
    log_file.write(f"\n[{started_at}] START {' '.join(command)}\n")
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
    return_code = process.wait()
    log_file.write(f"[{datetime.now(timezone.utc).isoformat()}] EXIT {return_code}\n")
if return_code:
    raise subprocess.CalledProcessError(return_code, command)
print(f"E1 complete in {(monotonic() - started) / 60:.1f} minutes")
print(f"Log: {LOG_PATH}")

## 6. Inspect and verify the outputs

The metric table contains absolute held-out results. The paired table contains literal left-minus-right differences; negative log-loss and Brier-score differences favor the model with hidden features.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

RESULT_DIR = OUTPUT_DIR / "conditional_hidden_state"
required_outputs = (
    "metrics.csv",
    "validation_selection.csv",
    "test_predictions.csv",
    "paired_differences.csv",
    "feature_blocks.json",
    "resolved_config.json",
    "result_record.md",
    "summary.json",
)
missing_outputs = [name for name in required_outputs if not (RESULT_DIR / name).exists()]
if missing_outputs:
    raise RuntimeError(f"E1 completed without required outputs: {missing_outputs}")

metrics = pd.read_csv(RESULT_DIR / "metrics.csv")
paired = pd.read_csv(RESULT_DIR / "paired_differences.csv")
summary = json.loads((RESULT_DIR / "summary.json").read_text())
if summary.get("status") != "complete":
    raise RuntimeError(f"Unexpected E1 status: {summary.get('status')!r}")
if set(metrics["condition"]) != {"N", "N+H", "O", "O+H", "H"}:
    raise RuntimeError("The metric artifact does not contain all five conditions.")

metric_columns = [
    "condition",
    "auroc",
    "average_precision",
    "log_loss",
    "brier_score",
    "error_exact",
    "correct_rejection",
    "process_f1",
    "error_within_1_accuracy",
    "error_within_2_accuracy",
    "complete_accuracy",
]
display(metrics[metric_columns].round(4))
display(
    paired[
        paired["metric"].isin(
            ["auroc", "average_precision", "log_loss", "brier_score", "process_f1"]
        )
    ][
        [
            "comparison",
            "metric",
            "estimate",
            "ci_low",
            "ci_high",
            "favorable_direction",
        ]
    ].round(4)
)
display(Markdown((RESULT_DIR / "result_record.md").read_text()))
print(f"Resolved config hash: {summary['config_sha256']}")
print(f"Saved outputs: {RESULT_DIR}")